# AutoML Quickstart: AutoGluon and PyCaret

Use AutoML as a budgeted baseline while preserving a held-out test set and externalizing artifacts.

- **Study time:** 30 minutes to review; runtime depends on the chosen time budget
- **Prerequisites:** tabular train/test workflow and metric selection
- **Mode:** `heavy`
- **Data policy:** no downloads; synthetic tabular data; model artifacts resolve to the external data root; AutoGluon and PyCaret use separate external environments; execution is opt-in with RUN_AUTOML=1
- **Provenance:** consolidated from the legacy AutoGluon and PyCaret teaching notebooks; errorful exploratory cells removed

Output convention: every retained textual result begins with a label that identifies the operation that produced it.


In [ ]:
import sys
from pathlib import Path


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Run this notebook from inside the DataCoding project")


PROJECT_ROOT = find_project_root()
source_dir = str(PROJECT_ROOT / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)

In [ ]:
import importlib.util
import os

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from datacoding.config import external_path

rng = np.random.default_rng(81)


def show(label, value):
    print(f"\n--- {label} ---\n{value}")


n_rows = 600
data = pd.DataFrame(
    {
        "age": rng.integers(18, 75, size=n_rows),
        "income": rng.normal(65_000, 18_000, size=n_rows),
        "channel": rng.choice(["web", "app", "store"], size=n_rows),
    }
)
logit = -3.0 + 0.00004 * data["income"] + 0.018 * data["age"] + (data["channel"] == "app") * 0.4
probability = 1 / (1 + np.exp(-logit))
data["label"] = (rng.random(n_rows) < probability).astype(int)
train_data, test_data = train_test_split(
    data,
    test_size=0.25,
    random_state=42,
    stratify=data["label"],
)

has_autogluon = importlib.util.find_spec("autogluon") is not None
has_pycaret = importlib.util.find_spec("pycaret") is not None
run_automl = os.environ.get("RUN_AUTOML") == "1"

show("Dataset | train/test shapes", (train_data.shape, test_data.shape))
show(
    "Optional stack | availability",
    {"AutoGluon": has_autogluon, "PyCaret": has_pycaret, "RUN_AUTOML": run_automl},
)

## 1. AutoGluon with an explicit time budget and external model path


In [ ]:
def run_autogluon(train_frame, test_frame, time_limit=120):
    from autogluon.tabular import TabularPredictor

    model_path = external_path("models", "autogluon_tabular_demo")
    predictor = TabularPredictor(
        label="label",
        eval_metric="f1",
        path=str(model_path),
    )
    predictor.fit(
        train_data=train_frame,
        time_limit=time_limit,
        presets="medium_quality",
    )
    return predictor, predictor.evaluate(test_frame), predictor.leaderboard(test_frame)


if run_automl and has_autogluon:
    autogluon_predictor, autogluon_score, autogluon_leaderboard = run_autogluon(
        train_data, test_data
    )
    show("AutoGluon | held-out metrics", autogluon_score)
    show("AutoGluon | leaderboard head", autogluon_leaderboard.head().to_string(index=False))
else:
    show(
        "AutoGluon | execution status",
        "skipped; activate the autogluon environment and set RUN_AUTOML=1",
    )

## 2. PyCaret object-oriented experiment API


In [ ]:
def run_pycaret(train_frame, test_frame):
    from pycaret.classification import ClassificationExperiment

    experiment = ClassificationExperiment()
    experiment.setup(
        data=train_frame,
        target="label",
        session_id=42,
        html=False,
        verbose=False,
    )
    best_model = experiment.compare_models(turbo=True)
    finalized_model = experiment.finalize_model(best_model)
    predictions = experiment.predict_model(finalized_model, data=test_frame)
    return experiment, finalized_model, predictions


if run_automl and has_pycaret:
    pycaret_experiment, pycaret_model, pycaret_predictions = run_pycaret(train_data, test_data)
    show("PyCaret | selected model", pycaret_model)
    show("PyCaret | held-out prediction columns", pycaret_predictions.columns.tolist())
else:
    show(
        "PyCaret | execution status",
        "skipped; activate the pycaret environment and set RUN_AUTOML=1",
    )

## 3. Fair-comparison checklist


In [ ]:
checklist = [
    "same train/test definition as the manual baseline",
    "metric selected before model comparison",
    "time and compute budget recorded",
    "test set excluded from selection",
    "leaderboard failures and latency inspected",
    "models and logs stored outside the vault",
]
show(
    "AutoML | review checklist",
    "\n".join(f"{index}. {item}" for index, item in enumerate(checklist, start=1)),
)